In [1]:
from tts.config.stage1.data_config import DataConfig
from tts.data.tts_datafactory import TTSDataFactory

data_config = DataConfig()


print("📦 Initializing datasets...")
data_factory = TTSDataFactory(data_config)

train_loader = data_factory.train_loader
valid_loader = data_factory.valid_loader

train_iter = iter(train_loader)

print(f"train batches: {len(train_loader)}")
print(f"valid batches: {len(valid_loader)}")

/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📦 Initializing datasets...
[Info] Parsed 149736 items from libritts
[Info] Filtering items by duration (2.5s ~ 25.0s)...


Filtering data: 100%|██████████| 149736/149736 [01:54<00:00, 1303.09it/s]


[Info] Filtered 41394 items. Remaining: 108342
[Info] Data split complete: 107259 train, 1083 valid
train batches: 17870
valid batches: 181


In [2]:
def summarize_batch(batch, batch_idx: int | None = None):
    x_lengths = batch.text_lengths
    y_lengths = batch.spec_lengths

    x = x_lengths.detach().cpu()
    y = y_lengths.detach().cpu()

    ratio = y.float() / x.clamp_min(1).float()

    B = int(x.numel())
    T_text_max = int(x.max().item())
    T_mel_max = int(y.max().item())

    pair_area = B * T_mel_max * T_text_max

    text_pad_total = B * T_text_max
    text_real_total = int(x.sum().item())
    text_pad_ratio = 1.0 - (text_real_total / max(text_pad_total, 1))

    mel_pad_total = B * T_mel_max
    mel_real_total = int(y.sum().item())
    mel_pad_ratio = 1.0 - (mel_real_total / max(mel_pad_total, 1))

    per_item_pair = x * y

    title = f"[batch {batch_idx}]" if batch_idx is not None else "[batch]"

    print("=" * 120)
    print(title)
    print(f"B={B}")
    print(f"T_text_max={T_text_max}")
    print(f"T_mel_max={T_mel_max}")
    print(f"B*T_mel*T_text={pair_area:,}")
    print("-" * 120)

    print("x_lengths:", x.tolist())
    print("y_lengths:", y.tolist())
    print("ratio y/x:", [round(v, 3) for v in ratio.tolist()])

    print("-" * 120)
    print(
        "x stats | "
        + f"min={int(x.min())}, "
        + f"mean={float(x.float().mean()):.2f}, "
        + f"median={float(x.float().median()):.2f}, "
        + f"max={int(x.max())}"
    )
    print(
        "y stats | "
        + f"min={int(y.min())}, "
        + f"mean={float(y.float().mean()):.2f}, "
        + f"median={float(y.float().median()):.2f}, "
        + f"max={int(y.max())}"
    )
    print(
        "ratio stats | "
        + f"min={float(ratio.min()):.3f}, "
        + f"mean={float(ratio.mean()):.3f}, "
        + f"median={float(ratio.median()):.3f}, "
        + f"max={float(ratio.max()):.3f}"
    )
    print(
        "per-item pair x*y | "
        + f"min={int(per_item_pair.min())}, "
        + f"mean={float(per_item_pair.float().mean()):.1f}, "
        + f"max={int(per_item_pair.max())}"
    )

    print("-" * 120)
    print(f"text padding waste: {text_pad_ratio * 100:.2f}%")
    print(f"mel padding waste:  {mel_pad_ratio * 100:.2f}%")

    suspicious = []
    for i, (xl, yl, r) in enumerate(zip(x.tolist(), y.tolist(), ratio.tolist())):
        flags = []
        if yl < xl:
            flags.append("BAD: y_len < x_len")
        if r < 1.05:
            flags.append("low_ratio")
        if r > 10.0:
            flags.append("high_ratio")
        if flags:
            suspicious.append((i, xl, yl, round(r, 3), flags))

    if suspicious:
        print("-" * 120)
        print("⚠️ suspicious samples:")
        for item in suspicious:
            print(item)

    if hasattr(batch, "scripts"):
        print("-" * 120)
        print("texts:")
        for i, s in enumerate(batch.scripts):
            print(f"[{i}] x={int(x[i])}, y={int(y[i])}, ratio={float(ratio[i]):.3f} | {s[:200]}")

    print("=" * 120)

In [3]:
batch_idx = 0

In [20]:
batch = next(train_iter)

batch_idx += 1
summarize_batch(batch, batch_idx=batch_idx)

[batch 17]
B=6
T_text_max=120
T_mel_max=528
B*T_mel*T_text=380,160
------------------------------------------------------------------------------------------------------------------------
x_lengths: [120, 116, 83, 95, 112, 104]
y_lengths: [528, 518, 471, 474, 485, 479]
ratio y/x: [4.4, 4.466, 5.675, 4.989, 4.33, 4.606]
------------------------------------------------------------------------------------------------------------------------
x stats | min=83, mean=105.00, median=104.00, max=120
y stats | min=471, mean=492.50, median=479.00, max=528
ratio stats | min=4.330, mean=4.744, median=4.466, max=5.675
per-item pair x*y | min=39093, mean=51951.2, max=63360
------------------------------------------------------------------------------------------------------------------------
text padding waste: 12.50%
mel padding waste:  6.72%
------------------------------------------------------------------------------------------------------------------------
texts:
[0] x=120, y=528, ratio=4.400 |